# Systematic uncertainty breakdown (final selected variables)

This notebook loads the precomputed systematic covariance files written by the workflow drivers:

- `run_syst_cosmics_chunked.sh` -> `Cosmics/cosmics_syst_dict.npz`
- `run_syst_multisim_chunked.sh` -> `MCstat/`, `Flux/`, `G4/`
- `run_syst_detvar_chunked.sh` -> `Detector/detector_syst_dict.npz`
- `run_syst_genie_chunked.sh` (seeded/produced elsewhere) -> `GENIE/cov_mat_dict.pkl`

It then computes, **for each final selected variable**:

1. **Per-category fractional uncertainty breakdown** (same style as the flux-by-knob example): Flux, G4, Detector (per WireMod/calo tag where available), and GENIE — as **two separate figures** for **rate** and **xsec** (each: per-knob curves plus category total).
2. **Category totals + summed covariance** as **two separate single-panel figures**: (a) per-category curves for Flux, G4, Detector, and **GENIE rate (total)** plus **sqrt(diag(C_Flux + C_G4 + C_Detector + C_GENIE_rate))**; (b) the same with **GENIE xsec (total)** and **sqrt(diag(C_Flux + C_G4 + C_Detector + C_GENIE_xsec))**. Saved as `syst_break_totals_rate__*` and `syst_break_totals_xsec__*`.
3. (Planned elsewhere in this notebook) A **summary** of integrated-rate uncertainty contributions and **top bin drivers** per source.

## Notes on the decomposition
The integrated-rate fractional variance for a variable with `n` bins is:

V = 1^T C 1

where `C` is the source fractional covariance matrix and `1` is an `n`-vector of ones.

For ranking “top drivers” we compute the per-bin signed variance contributions:

t_i = (C * 1)_i

and rank by `|t_i|`.


In [ ]:
import os
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.dataset_locations import default_syst_disk_root, PLOTS_BASE
from analysis_village.numucc_1p0pi.final_selected_evt_vars import (
    CORE_SELECTED_EVT_VARIABLE_CONFIGS,
    with_final_selected_evt_variables,
)
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig

from analysis_village.numucc_1p0pi import utils as _numu_utils  # presentation.mplstyle, dpi


In [ ]:
PLOTS_BASE

In [ ]:
# ----------------------------
# User configuration
# ----------------------------

# Root of the syst_disk_layout tree. Must contain:
#   MCstat/mcstat_syst_dict.npz
#   Flux/flux_syst_dict.npz
#   G4/g4_syst_dict.npz
#   Cosmics/cosmics_syst_dict.npz
#   Detector/detector_syst_dict.npz
#   GENIE/cov_mat_dict.pkl
# SYST_DISK_ROOT = os.environ.get('NUMUCC_SYST_DISK_ROOT', None)
SYST_DISK_ROOT = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final"
if not SYST_DISK_ROOT:
    SYST_DISK_ROOT = str(default_syst_disk_root())
SYST_DISK_ROOT = str(Path(SYST_DISK_ROOT).expanduser())

# Subset of variables to plot (None = all final selected)
# Provide strings like 'integrated', 'muon-p', etc.
VARS_TO_PLOT = None

# Output directory for figures
OUT_DIR = Path(PLOTS_BASE) / 'syst_uncertainty_breakdown' / 'final_selected_pretty'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If True, run the last cell to write Flux/G4/Detector/GENIE (rate + xsec as separate PNGs)/totals for every variable in ``var_configs`` (slow).
SAVE_ALL_BREAKDOWN_PNGS = False

# Sources to include
SOURCES = ['Flux', 'G4', 'detvar', 'cosmic', 'GENIE', 'MCstat']

# Top-k bin drivers per source
TOP_K = 5

print('SYST_DISK_ROOT =', SYST_DISK_ROOT)
print('OUT_DIR       =', str(OUT_DIR))


In [ ]:
# print all subdirectories in SYST_DISK_ROOT
print(os.listdir(SYST_DISK_ROOT))


In [ ]:
# ----------------------------
# Variable catalogue
# ----------------------------

# var_configs = with_final_selected_evt_variables(list(CORE_SELECTED_EVT_VARIABLE_CONFIGS))

var_configs = [
                VariableConfig.all_events(),
                VariableConfig.muon_momentum(),
                VariableConfig.muon_direction(),
                VariableConfig.proton_momentum(),
                VariableConfig.proton_direction(),
                VariableConfig.tki_del_alpha(),
                VariableConfig.tki_del_phi(),
                VariableConfig.tki_del_Tp(),
                VariableConfig.tki_del_p(),
                VariableConfig.tki_del_Tp_x(),
                VariableConfig.tki_del_Tp_y(),
                # VariableConfig.muon_direction_x(),
                # VariableConfig.muon_direction_y(),
                # VariableConfig.proton_direction_x(),
                # VariableConfig.proton_direction_y(),
                # VariableConfig.opening_angle(),
                # VariableConfig.vertex_x(),
                # VariableConfig.vertex_y(),
                # VariableConfig.vertex_z(),
                ]

if VARS_TO_PLOT is not None:
    allowed = set(VARS_TO_PLOT)
    var_configs = [vc for vc in var_configs if vc.var_save_name in allowed]

var_names = [vc.var_save_name for vc in var_configs]
print(f'Final selected variables: {len(var_names)}')
print(' - ' + ', '.join(var_names))


In [ ]:
# ----------------------------
# Load covariance payloads
# ----------------------------

from analysis_village.numucc_1p0pi.syst_disk_layout import syst_disk_paths

paths = syst_disk_paths(SYST_DISK_ROOT)

needed = {
    'flux_npz': paths['flux'],
    'g4_npz': paths['g4'],
    'cosmics_npz': paths['cosmics'],
}

missing = [k for k, p in needed.items() if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required syst files:\n" + '\n'.join(f"  {k}: {needed[k]}" for k in missing)
    )

flux_npz = np.load(needed['flux_npz'], allow_pickle=True)
g4_npz = np.load(needed['g4_npz'], allow_pickle=True)
cosmics_npz = np.load(needed['cosmics_npz'], allow_pickle=True)

mcstat_npz = None
_mcstat_path = paths['mcstat']
if Path(_mcstat_path).is_file():
    mcstat_npz = np.load(_mcstat_path, allow_pickle=True)
    print('Loaded MCstat:', _mcstat_path)
else:
    print('MCstat NPZ not found (MCstat omitted from category summary):', _mcstat_path)


def print_genie_pickle_key_summary(genie_pkl):
    """Print all keys in ``GENIE/cov_mat_dict.pkl`` for debugging."""
    if genie_pkl is None:
        print("GENIE pickle not loaded")
        return
    top = sorted(genie_pkl.keys())
    print(f"GENIE cov_mat_dict.pkl — top-level variable keys ({len(top)}):")
    for k in top:
        print(f"  {k}")
    inner = set()
    for vsn, cell in genie_pkl.items():
        if isinstance(cell, dict):
            inner.update(cell.keys())
        else:
            inner.add(f"<non-dict {type(cell).__name__}>")
    inner_sorted = sorted(inner)
    print(f"GENIE pickle — union of per-variable sub-keys ({len(inner_sorted)}):")
    for k in inner_sorted:
        print(f"  {k}")


genie_blob = None
_genie_path = paths['genie']
if Path(_genie_path).is_file():
    with open(_genie_path, 'rb') as gf:
        genie_blob = pickle.load(gf)
    print('Loaded GENIE:', _genie_path)
    print_genie_pickle_key_summary(genie_blob)
else:
    print('GENIE pickle not found (GENIE combined-from-pickle plots skipped):', _genie_path)

wiremod_npz = None
_wm_path = Path(SYST_DISK_ROOT) / 'WireMod' / 'Detector' / 'detector_syst_dict.npz'
if _wm_path.is_file():
    wiremod_npz = np.load(_wm_path, allow_pickle=True)
    print('Loaded WireMod detector:', _wm_path)
else:
    print('WireMod detector NPZ not found (WireMod plots skipped):', _wm_path)

sce_npz = None
_sce_path = Path(SYST_DISK_ROOT) / 'SCE' / 'Detector' / 'detector_syst_dict.npz'
if _sce_path.is_file():
    sce_npz = np.load(_sce_path, allow_pickle=True)
    print('Loaded SCE detector:', _sce_path)
else:
    print('SCE detector NPZ not found (SCE plots skipped):', _sce_path)

# Legacy combined detector file (optional; breakdown uses WireMod / SCE separately)
detector_npz = None
if Path(paths['detector']).is_file():
    detector_npz = np.load(paths['detector'], allow_pickle=True)
    print('Loaded combined Detector:', paths['detector'])

print('Loaded syst covariance payloads.')


In [ ]:
# ----------------------------
# Helpers: fractional unc from cov_frac and per-category breakdown plots
# (axis style matches ``unfolding-data.ipynb`` ``get_syst_unc(..., plot=True)``.)
# ----------------------------

# Fixed size/aspect for every per-category breakdown figure (display + saved PNG).
BREAKDOWN_FIGSIZE = (7.0, 6.0)
BREAKDOWN_FIG_DPI = 100  # on-screen display dpi (figures are saved, not shown)
BREAKDOWN_SAVE_DPI = int(_numu_utils.dpi)  # file output (typically 300)
BREAKDOWN_SUBPLOT = dict(left=0.11, right=0.99, top=0.94, bottom=0.40)
LEGEND_OUTSIDE_THRESHOLD = 20
BREAKDOWN_SUBPLOT_ONPLOT = dict(left=0.11, right=0.99, top=0.94, bottom=0.11)

# Keep a single canvas size/dpi for all plots in this notebook.
plt.rcParams["figure.figsize"] = BREAKDOWN_FIGSIZE
plt.rcParams["figure.dpi"] = BREAKDOWN_FIG_DPI

from analysis_village.numucc_1p0pi.utils import add_approval_text, add_genie_version_text

APPROVAL_TEXT = "preliminary"  # "internal" | "preliminary" | anything else -> off


def _lock_breakdown_figure_size(fig):
    """Force every breakdown figure to the same canvas (inches + screen dpi)."""
    fig.set_size_inches(BREAKDOWN_FIGSIZE, forward=False)
    fig.set_dpi(BREAKDOWN_FIG_DPI)


def make_breakdown_figure():
    """Return (fig, ax) with standard breakdown size (legend placement set in ``style_uncertainty_axis``)."""
    fig = plt.figure(figsize=BREAKDOWN_FIGSIZE, dpi=BREAKDOWN_FIG_DPI)
    ax = fig.add_subplot(111)
    fig.subplots_adjust(**BREAKDOWN_SUBPLOT_ONPLOT)
    return fig, ax


def save_breakdown_figure(fig, out_path):
    """Save PNG + PDF with tight bounding box (no show/close — safe for batch saves)."""
    _lock_breakdown_figure_size(fig)
    out_path = Path(out_path)
    pdf_path = out_path.with_suffix(".pdf")
    save_kwargs = dict(
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    fig.savefig(out_path, dpi=BREAKDOWN_SAVE_DPI, **save_kwargs)
    fig.savefig(pdf_path, **save_kwargs)





def frac_unc_diag(cov_frac):
    c = np.asarray(cov_frac, dtype=np.float64)
    return np.nan_to_num(np.sqrt(np.diag(c)), nan=0.0, posinf=0.0, neginf=0.0)


def frac_unc_pct(cov_frac):
    """Per-bin fractional uncertainty as percent (100 × sqrt(diag(cov_frac)))."""
    return 100.0 * frac_unc_diag(cov_frac)


def integrated_rate_frac_variance(cov_frac):
    """Integrated-rate fractional variance V = 1^T C 1 (single-bin: ``C[0,0]``)."""
    c = np.asarray(cov_frac, dtype=np.float64)
    ones = np.ones(c.shape[0], dtype=np.float64)
    return float(ones @ c @ ones)


def _integrated_knob_scores(by_knob):
    if not by_knob:
        return {}
    return {
        kn: integrated_rate_frac_variance(pack["cov_frac"])
        for kn, pack in by_knob.items()
    }


def _integrated_matrix_scores(matrix_by_key):
    if not matrix_by_key:
        return {}
    return {
        kn: integrated_rate_frac_variance(m)
        for kn, m in matrix_by_key.items()
    }


def _sort_keys_by_score(keys, scores):
    keys = list(keys)
    if not scores:
        return sorted(keys)
    return sorted(keys, key=lambda k: scores.get(k, 0.0), reverse=True)


def _top_knob_keys(matrix_by_key, scores, k=10):
    """Top *k* knob keys by integrated-rate fractional variance (for combined GENIE plots)."""
    if not matrix_by_key:
        return []
    sc = scores if scores is not None else _integrated_matrix_scores(matrix_by_key)
    return _sort_keys_by_score(matrix_by_key.keys(), sc)[:k]


def _colors_from_cmap(name):
    cmap = plt.get_cmap(name)
    if hasattr(cmap, "colors"):
        return list(cmap.colors)
    ncols = getattr(cmap, "N", 256)
    return [cmap(i / max(ncols - 1, 1)) for i in range(ncols)]


# tab10, then Set2 (matplotlib qualitative maps).
_STEP_COLOR_POOL = _colors_from_cmap("tab10") + _colors_from_cmap("Set2")


def _uncertainty_step_colors(n):
    """Line colors: all ``tab10`` swatches, then all ``Set2`` swatches."""
    if n <= 0:
        return []
    if n <= len(_STEP_COLOR_POOL):
        return _STEP_COLOR_POOL[:n]
    out = list(_STEP_COLOR_POOL)
    k = 0
    while len(out) < n:
        out.append(_STEP_COLOR_POOL[k % len(_STEP_COLOR_POOL)])
        k += 1
    return out[:n]


def _reserve_axes_space_for_top_legend(ax, legend, pad=0.02):
    """Expand the y upper limit so curve data stays below a top-center legend."""
    fig = ax.get_figure()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    leg_bbox = legend.get_window_extent(renderer).transformed(ax.transAxes.inverted())
    legend_bottom = leg_bbox.y0 - pad
    if legend_bottom < 0.99:
        ylo, yhi = ax.get_ylim()
        if legend_bottom > 0.05:
            ax.set_ylim(ylo, yhi / legend_bottom)


def _add_top_center_legend(ax, handles, labels, ncol, fontsize):
    ncol = max(1, min(ncol, len(labels)))
    leg = ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=ncol,
        fontsize=fontsize,
        frameon=True,
        borderaxespad=0.35,
    )
    _reserve_axes_space_for_top_legend(ax, leg)

    # Add GENIE/approval text just below the legend box.
    fig = ax.get_figure()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    leg_bbox = leg.get_window_extent(renderer).transformed(ax.transAxes.inverted())
    x = 0.5
    ha = "center"
    y0 = max(float(leg_bbox.y0) - 0.01, 0.0)
    add_approval_text(APPROVAL_TEXT, x, y0-0.03, ha) #, fontsize=1)
    add_genie_version_text(x, max(y0 - 0.1, 0.0), ha)

    return leg


def style_uncertainty_axis(ax, var_config, pct_series_list, legend_ncol=3, legend_fontsize=10):
    """Grid, limits, and top-center legend (outside bottom if > ``LEGEND_OUTSIDE_THRESHOLD`` entries)."""
    flat = [np.asarray(s, dtype=np.float64).ravel() for s in pct_series_list if s is not None and len(s)]
    ymax = max((np.nanmax(np.where(np.isfinite(x), x, 0.0)) for x in flat), default=0.0)
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    # ax.set_ylim(0, 14)
    xlab = var_config.var_labels[1] if getattr(var_config, "var_labels", None) else var_config.var_save_name
    ax.set_xlabel(xlab)
    ax.set_ylabel("Uncertainty [%]")
    ax.grid(which="major", linestyle="-", linewidth=0.7, alpha=0.7)
    ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.5)
    ax.minorticks_on()
    if getattr(var_config, "var_save_name", None) == "integrated":
        ax.set_xticks([])
        ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    fig = ax.get_figure()
    handles, labels = ax.get_legend_handles_labels()
    if labels and len(labels) > LEGEND_OUTSIDE_THRESHOLD:
        fig.subplots_adjust(**BREAKDOWN_SUBPLOT)
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, BREAKDOWN_SUBPLOT["bottom"] * 0.42),
            ncol=legend_ncol,
            fontsize=legend_fontsize,
            frameon=True,
            borderaxespad=0.0,
        )
    elif labels:
        fig.subplots_adjust(**BREAKDOWN_SUBPLOT_ONPLOT)
        _add_top_center_legend(ax, handles, labels, legend_ncol, legend_fontsize)
    else:
        fig.subplots_adjust(**BREAKDOWN_SUBPLOT_ONPLOT)
    _lock_breakdown_figure_size(fig)





def flux_knob_display_label(knob_name):
    """Matplotlib legend label for BNB flux multisim keys (``makedf.bnbsyst.regen_systematics``)."""
    kn = str(knob_name)
    base = kn[:-5] if kn.endswith("_Flux") else kn
    labels = {
        "expskin": r"Exp. skin",
        "horncurrent": r"Horn current",
        "kminus": r"$K^-$",
        "kplus": r"$K^+$",
        "kzero": r"$K^0$",
        "piminus": r"$\pi^-$",
        "piplus": r"$\pi^+$",
        "pioninexsec": r"$\pi$ inel. $\sigma$",
        "pionqexsec": r"$\pi$ QE $\sigma$",
        "piontotxsec": r"$\pi$ total $\sigma$",
        "nucleoninexsec": r"Nucleon inel. $\sigma$",
        "nucleonqexsec": r"Nucleon QE $\sigma$",
        "nucleontotxsec": r"Nucleon total $\sigma$",
    }
    return labels.get(base, base.replace("_", " "))


_G4_KNOB_LABELS = {
    "neutron": r"$n$",
    "piminus": r"$\pi^-$",
    "piplus": r"$\pi^+$",
    "proton": r"$p$",
    "kminus": r"$K^-$",
    "kplus": r"$K^+$",
}


def _g4_knob_particle_key(knob_name):
    """Normalize G4 knob id to particle key (``neutron``, ``piminus``, ...)."""
    kn = str(knob_name).strip()
    if kn.endswith("_Geant4"):
        kn = kn[: -len("_Geant4")]
    if kn.startswith("reinteractions_"):
        kn = kn[len("reinteractions_") :]
    return kn


def g4_knob_display_label(knob_name):
    """Matplotlib legend label for Geant4 reinteraction multisim keys."""
    return _G4_KNOB_LABELS.get(_g4_knob_particle_key(knob_name))


def syst_knob_display_label(knob_name):
    """Legend label for flux or G4 per-knob breakdown keys."""
    kn = str(knob_name)
    if kn.endswith("_Flux"):
        return flux_knob_display_label(kn)
    g4_lab = g4_knob_display_label(kn)
    if g4_lab is not None:
        return g4_lab
    return kn.replace("_", " ")


_GENIE_KNOB_STRIP_PREFIXES = (
    "GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_",
    "CCQETemplateReweight_SBNNuSyst_multisigma_",
    "QEInterference_SBNNuSyst_multisigma_",
    "ZExpPCAWeighter_SBNNuSyst_multisigma_",
    "GENIEReWeight_SBNNuSyst_multisigma_",
    "GENIEReWeight_SBN_v1_multisim_",
    "GENIEReWeight_SBN_v1_",
    "GENIEReWeight_",
)

_GENIE_KNOB_STRIP_TOKENS = (
    "SBNNuSyst_",
    "EDepFSI_",
    "SBNNuSyst",
    "EDepFSI",
    "multisigma",
    "multisim",
)


def genie_knob_display_label(knob_name):
    """Short GENIE multisim legend labels (drop version / multisim boilerplate)."""
    kn = str(knob_name)
    for prefix in _GENIE_KNOB_STRIP_PREFIXES:
        if kn.startswith(prefix):
            kn = kn[len(prefix) :]
            break
    for _rm in _GENIE_KNOB_STRIP_TOKENS:
        kn = kn.replace(_rm, "")
    kn = kn.strip("_")
    while "__" in kn:
        kn = kn.replace("__", "_")
    return kn.replace("_", " ")


def plot_knob_breakdown(
    ax,
    var_config,
    by_knob,
    total_cov_frac,
    title="",
    label_fn=None,
    knob_sort_scores=None,
    legend_ncol=3,
):
    bins = var_config.bins
    bc = var_config.bin_centers
    label_fn = label_fn or syst_knob_display_label
    pct_list = []
    if by_knob:
        if knob_sort_scores is None and getattr(var_config, "var_save_name", None) == "integrated":
            knob_sort_scores = _integrated_knob_scores(by_knob)
        knob_keys = _sort_keys_by_score(by_knob.keys(), knob_sort_scores)
        colors = _uncertainty_step_colors(len(knob_keys))
        for kn, color in zip(knob_keys, colors):
            cf = by_knob[kn]["cov_frac"]
            w = frac_weights_for_plot(cf, var_config)
            pct_list.append(w)
            ax.hist(
                bc,
                bins=bins,
                weights=w,
                histtype="step",
                linewidth=2,
                color=color,
                label=label_fn(kn),
            )
    wtot = frac_weights_for_plot(total_cov_frac, var_config)
    pct_list.append(wtot)
    ax.hist(bc, bins=bins, weights=wtot, histtype="step", linewidth=2, color="k", label="Total")
    style_uncertainty_axis(ax, var_config, pct_list, legend_ncol=legend_ncol)


def cosmics_var_pack(cosmics_npz, var_name):
    """Return the ``Cosmics`` payload dict for one variable slug (no per-knob nesting)."""
    if cosmics_npz is None or var_name not in cosmics_npz:
        return None
    cell = cosmics_npz[var_name].item()
    if isinstance(cell, dict) and "Cosmics" in cell:
        return cell["Cosmics"]
    return cell if isinstance(cell, dict) else None


def cosmics_template_cov_frac(cosmics_npz, var_name):
    """Cosmic-template fractional covariance (``Cosmics`` pack), same path as ``utils.get_syst_unc``."""
    pay = cosmics_var_pack(cosmics_npz, var_name)
    if pay is None:
        return None
    if pay.get("cov_frac") is not None:
        return np.asarray(pay["cov_frac"], dtype=np.float64)
    rate = pay.get("rate")
    if isinstance(rate, dict) and rate.get("cov_frac") is not None:
        return np.asarray(rate["cov_frac"], dtype=np.float64)
    return None


def cosmics_selected_rate_cov_frac(cosmics_npz, var_name):
    """Fractional covariance on selected event rate (contamination-scaled; ``systematics-cosmic.ipynb``)."""
    if cosmics_npz is None or var_name not in cosmics_npz:
        return None
    cell = cosmics_npz[var_name].item()
    if isinstance(cell, dict) and "SelectedRate" in cell:
        rate = cell["SelectedRate"].get("rate")
        if isinstance(rate, dict) and "cov_frac" in rate:
            return np.asarray(rate["cov_frac"], dtype=np.float64)
    return None


def cosmics_cov_frac(cosmics_npz, var_name):
    """Contamination-scaled cosmic uncertainty on selected rate (``SelectedRate`` in NPZ)."""
    return cosmics_selected_rate_cov_frac(cosmics_npz, var_name)


def cosmics_plot_weights(cosmics_npz, var_config):
    """Per-bin uncertainty [%] for cosmics (selected rate; category + totals share this)."""
    cov = cosmics_cov_frac(cosmics_npz, var_config.var_save_name)
    if cov is None:
        return None
    return frac_weights_for_plot(cov, var_config)


def frac_weights_for_plot(cov_frac, var_config):
    """Per-bin uncertainty [%] for breakdown plots (aligns length; integrated uses flat max)."""
    w = frac_unc_pct(cov_frac)
    n = len(var_config.bin_centers)
    if len(w) != n:
        if len(w) == 1 and n > 1:
            w = np.full(n, float(w[0]))
        elif len(w) > n:
            w = np.asarray(w[:n], dtype=np.float64)
        elif len(w) < n:
            out = np.zeros(n, dtype=np.float64)
            out[: len(w)] = w
            w = out
    if getattr(var_config, "var_save_name", None) == "integrated" and len(w):
        w = np.full(n, float(np.nanmax(w)))
    return w


def data_stat_frac_pct(data_evt_df, var_config, scale=1.0):
    """Per-bin data Poisson fractional uncertainty [%] (``unfolding-data.ipynb`` / ``get_syst_unc`` plot)."""
    col = var_config.var_evt_reco_col
    n_data, _ = np.histogram(data_evt_df[col], bins=var_config.bins)
    n_data = np.asarray(n_data, dtype=np.float64)
    n_scaled = n_data * scale
    with np.errstate(divide="ignore", invalid="ignore"):
        n_data_err = np.sqrt(np.maximum(n_scaled, 0.0))
        frac = np.where(n_scaled > 0, n_data_err / n_scaled, 0.0)
    w = 100.0 * frac
    n = len(var_config.bin_centers)
    if len(w) != n:
        if len(w) == 1 and n > 1:
            w = np.full(n, float(w[0]))
        elif len(w) > n:
            w = np.asarray(w[:n], dtype=np.float64)
        elif len(w) < n:
            out = np.zeros(n, dtype=np.float64)
            out[: len(w)] = w
            w = out
    if getattr(var_config, "var_save_name", None) == "integrated" and len(w):
        w = np.full(n, float(np.nanmax(w)))
    return w


def cosmics_plot_label(cosmics_npz, var_name):
    pay = cosmics_var_pack(cosmics_npz, var_name)
    mode = pay.get("cv_mode") if isinstance(pay, dict) else None
    return f"Cosmics ({mode})" if mode else "Cosmics"


def plot_cosmics_breakdown(ax, var_config, cosmics_npz, title=None):
    """Cosmic uncertainty on selected event rate (contamination-scaled ``SelectedRate``)."""
    vsn = var_config.var_save_name
    w = cosmics_plot_weights(cosmics_npz, var_config)
    if w is None:
        ax.text(
            0.5, 0.5, "No SelectedRate cosmics data",
            transform=ax.transAxes, ha="center", va="center",
        )
        return
    color = _uncertainty_step_colors(1)[0]
    ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=w,
        histtype="step",
        linewidth=2,
        color=color,
        label=cosmics_selected_rate_plot_label(cosmics_npz, vsn),
    )
    style_uncertainty_axis(ax, var_config, [w])


def cosmics_selected_rate_plot_label(cosmics_npz, var_name):
    pay = cosmics_var_pack(cosmics_npz, var_name)
    mode = pay.get("cv_mode") if isinstance(pay, dict) else None
    if mode:
        return f"Cosmics on selected rate ({mode})"
    return "Cosmics on selected rate"


def plot_cosmics_selected_rate_breakdown(ax, var_config, cosmics_npz):
    """Alias for :func:`plot_cosmics_breakdown` (selected-rate cosmics)."""
    plot_cosmics_breakdown(ax, var_config, cosmics_npz)


def mcstat_cov_frac(mcstat_npz, var_name):
    """Fractional covariance from bundled MCstat multisim (``MCstat`` pack in NPZ)."""
    if mcstat_npz is None or var_name not in mcstat_npz:
        return None
    cell = mcstat_npz[var_name].item()
    if isinstance(cell, dict) and "MCstat" in cell:
        pay = cell["MCstat"]
        if isinstance(pay, dict) and pay.get("cov_frac") is not None:
            return np.asarray(pay["cov_frac"], dtype=np.float64)
    return None


def plot_mcstat_breakdown(ax, var_config, mcstat_npz, title=None):
    """Finite-MC statistical uncertainty (bundled MCstat multisim)."""
    cov = mcstat_cov_frac(mcstat_npz, var_config.var_save_name)
    if cov is None:
        ax.text(
            0.5, 0.5, "No MCstat data",
            transform=ax.transAxes, ha="center", va="center",
        )
        return
    w = frac_weights_for_plot(cov, var_config)
    color = _uncertainty_step_colors(1)[0]
    ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=w,
        histtype="step",
        linewidth=2,
        color=color,
        label="MC stat.",
    )
    style_uncertainty_axis(ax, var_config, [w])


def detector_tag_display_label(tag):
    labels = {
        "wiremodyz": "WireMod Y-Z",
        "wiremodxtxw": r"WireMod X-$\theta_{xw}$",
        "0xsce": "SCE off",
        "2xsce": "SCE twice",
        "sce0x": "SCE off",
        "sce2x": "SCE twice",
    }
    t = str(tag).lower().replace("-", "").replace("_", "")
    return labels.get(t, str(tag).replace("_", " "))


def load_genie_mode_npzs(syst_root):
    """Load all ``genie-<mode>/genie-<mode>_syst_dict.npz`` files under syst root."""
    return {
        mode: np.load(p, allow_pickle=True)
        for mode, p in discover_genie_mode_npz_paths(syst_root).items()
    }


def plot_detector_subsystem_breakdown(ax, var_config, det_npz, title=None):
    """Per-tag detector curves + combined total (WireMod or SCE ``detector_syst_dict.npz``)."""
    plot_detector_combined_breakdown(
        ax,
        var_config,
        wiremod_npz=det_npz,
        sce_npz=None,
        title=title,
    )


def plot_detector_combined_breakdown(
    ax,
    var_config,
    wiremod_npz=None,
    sce_npz=None,
    title=None,
):
    """WireMod + SCE detector knobs on one plot (independent tags, summed total)."""
    vsn = var_config.var_save_name
    bins = var_config.bins
    bc = var_config.bin_centers
    all_by_tag = {}
    totals = []

    for source, det_npz in (("WireMod", wiremod_npz), ("SCE", sce_npz)):
        if det_npz is None:
            continue
        by_tag, tot = detector_by_tag_dict(det_npz, vsn)
        if tot is not None:
            totals.append(tot)
        for tag, cf in (by_tag or {}).items():
            lab = detector_tag_display_label(tag)
            all_by_tag[lab] = cf

    pct_list = []
    if all_by_tag:
        int_scores = {}
        for source, det_npz in (("WireMod", wiremod_npz), ("SCE", sce_npz)):
            if det_npz is None:
                continue
            by_int, _ = detector_by_tag_dict(det_npz, "integrated")
            for tag, cf in (by_int or {}).items():
                lab = detector_tag_display_label(tag)
                int_scores[lab] = integrated_rate_frac_variance(cf)
        tag_keys = _sort_keys_by_score(all_by_tag.keys(), int_scores)
        colors = _uncertainty_step_colors(len(tag_keys))
        for lab, color in zip(tag_keys, colors):
            w = frac_weights_for_plot(all_by_tag[lab], var_config)
            pct_list.append(w)
            ax.hist(
                bc,
                bins=bins,
                weights=w,
                histtype="step",
                linewidth=2,
                color=color,
                label=lab,
            )

    if totals:
        comb = _sum_cov_frac_matrices(totals) if len(totals) > 1 else totals[0]
        wtot = frac_weights_for_plot(comb, var_config)
        pct_list.append(wtot)
        ax.hist(
            bc,
            bins=bins,
            weights=wtot,
            histtype="step",
            linewidth=2,
            color="k",
            label="Detector total",
        )

    if not pct_list:
        ax.text(0.5, 0.5, "No detector data", transform=ax.transAxes, ha="center", va="center")
        return

    style_uncertainty_axis(
        ax,
        var_config,
        pct_list,
        legend_ncol=2,
    )


def discover_genie_mode_npz_paths(syst_root):
    """``{CCQE, DIS, ...} -> path`` for ``genie-<mode>/genie-<mode>_syst_dict.npz`` under syst root."""
    root = Path(syst_root)
    out = {}
    for d in sorted(root.iterdir()):
        if not d.is_dir() or not d.name.startswith("genie-"):
            continue
        mode = d.name[len("genie-") :]
        p = d / f"genie-{mode}_syst_dict.npz"
        if p.is_file():
            out[mode] = p
    return out


def genie_group_by_knob(genie_mode_npz, var_name):
    if genie_mode_npz is None or var_name not in genie_mode_npz:
        return None
    cell = genie_mode_npz[var_name].item()
    return cell if isinstance(cell, dict) else None


def genie_group_knob_covs(by_knob):
    """Build a ``genie_pack`` from one interaction-mode NPZ entry (var-first layout)."""
    rate_parts, xsec_parts = {}, {}
    for kn, pack in sorted((by_knob or {}).items()):
        if not isinstance(pack, dict):
            continue
        if isinstance(pack.get("rate"), dict) and "cov_frac" in pack["rate"]:
            rate_parts[kn] = np.asarray(pack["rate"]["cov_frac"], dtype=np.float64)
        if isinstance(pack.get("xsec"), dict) and "cov_frac" in pack["xsec"]:
            xsec_parts[kn] = np.asarray(pack["xsec"]["cov_frac"], dtype=np.float64)
    rate_total = _sum_cov_frac_matrices(rate_parts.values()) if rate_parts else None
    xsec_total = _sum_cov_frac_matrices(xsec_parts.values()) if xsec_parts else None
    return {
        "rate_parts": rate_parts,
        "xsec_parts": xsec_parts,
        "rate_total": rate_total,
        "xsec_total": xsec_total,
    }


def genie_pickle_mode_pack(genie_blob, var_name, mode):
    """Per-interaction-mode ``genie_pack`` from ``GENIE/cov_mat_dict.pkl`` (``GENIE_GROUP_KNOBS``)."""
    from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS

    gd = genie_var_dict(genie_blob, var_name)
    if gd is None:
        return None
    rate_parts, xsec_parts = {}, {}
    for kn in GENIE_GROUP_KNOBS.get(mode, []):
        rk = f"{kn}_rate"
        if rk in gd and isinstance(gd[rk], np.ndarray) and np.asarray(gd[rk]).ndim == 2:
            rate_parts[kn] = np.asarray(gd[rk], dtype=np.float64)
        if kn in gd and isinstance(gd[kn], np.ndarray) and np.asarray(gd[kn]).ndim == 2:
            xsec_parts[kn] = np.asarray(gd[kn], dtype=np.float64)
    if not rate_parts and not xsec_parts:
        return None
    return {
        "rate_parts": rate_parts,
        "xsec_parts": xsec_parts,
        "rate_total": _sum_cov_frac_matrices(rate_parts.values()) if rate_parts else None,
        "xsec_total": _sum_cov_frac_matrices(xsec_parts.values()) if xsec_parts else None,
    }


def genie_knob_excluded_from_combined_breakdown(knob, mode):
    """Knobs omitted from the all-GENIE combined breakdown (not from per-mode plots)."""
    if mode == "Other" and (knob.endswith("_pi") or knob.endswith("_N")):
        return True
    if mode == "Ar23p":
        if "D_ZExp" in knob:
            return True
        if "q0bin5" in knob:
            return True
        if "EDepFSI_DecayAngMEC" in knob:
            return True
        if "EDepFSI_NormCCMEC" in knob:
            return True
    return False


def genie_combined_breakdown_group_key(knob):
    """Map a GENIE knob to a combined-plot family (bin / dial / ZExp-b groups)."""
    import re

    kn = str(knob)
    if re.search(r"_b\d+$", kn) and "ZExp" in kn:
        return "ZExp"
    m = re.search(r"_dial_\d+$", kn)
    if m:
        return kn[: m.start()].rsplit("_", 1)[-1]
    m = re.search(r"_q0bin\d+$", kn)
    if m:
        prefix = kn[: m.start()]
        if "Martini" in prefix:
            return "MEC Martini"
        if "Valenica" in prefix:
            return "MEC Valencia"
        return prefix.rsplit("_", 1)[-1]
    m = re.search(r"bin\d+$", kn)
    if m:
        prefix = kn[: m.start()]
        if "Martini" in prefix:
            return "MEC Martini"
        if "Valenica" in prefix:
            return "MEC Valencia"
        return prefix.rsplit("_", 1)[-1] if prefix else kn
    return kn


def _accumulate_genie_grouped_cov(parts, group_key, matrix):
    arr = np.asarray(matrix, dtype=np.float64)
    if group_key not in parts:
        parts[group_key] = arr.copy()
    else:
        parts[group_key] = parts[group_key] + arr


def genie_combined_group_display_label(group_key):
    """Legend label for a combined-breakdown family (short keys pass through)."""
    gk = str(group_key)
    if gk in ("CRPA", "SF", "ZExp", "QEIntf", "MEC Martini", "MEC Valencia"):
        return gk
    return genie_knob_display_label(gk)


def genie_pickle_combined_breakdown_pack(genie_blob, var_name):
    """All GENIE knobs across modes; exclusions + bin/dial/ZExp-b grouping."""
    from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS, GENIE_GROUP_ORDER

    gd = genie_var_dict(genie_blob, var_name)
    if gd is None:
        return None
    rate_parts, xsec_parts = {}, {}
    for mode in GENIE_GROUP_ORDER:
        for kn in GENIE_GROUP_KNOBS.get(mode, []):
            if genie_knob_excluded_from_combined_breakdown(kn, mode):
                continue
            gkey = genie_combined_breakdown_group_key(kn)
            rk = f"{kn}_rate"
            if rk in gd and isinstance(gd[rk], np.ndarray) and np.asarray(gd[rk]).ndim == 2:
                _accumulate_genie_grouped_cov(rate_parts, gkey, gd[rk])
            if kn in gd and isinstance(gd[kn], np.ndarray) and np.asarray(gd[kn]).ndim == 2:
                _accumulate_genie_grouped_cov(xsec_parts, gkey, gd[kn])
    if not rate_parts and not xsec_parts:
        return None
    return {
        "rate_parts": rate_parts,
        "xsec_parts": xsec_parts,
        "rate_total": _sum_cov_frac_matrices(rate_parts.values()) if rate_parts else None,
        "xsec_total": _sum_cov_frac_matrices(xsec_parts.values()) if xsec_parts else None,
    }


def merge_genie_modes_into_pickle(genie_blob, mode_npzs, modes):
    """Add per-knob matrices from ``genie-<mode>`` NPZs into the GENIE pickle (same layout as DIS/Other).

    Only adds knobs not already present. Updates ``genie`` / ``genie_rate`` totals. Returns number of
    knob×kind matrix blocks added.
    """
    from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS

    if genie_blob is None:
        return 0
    n_added = 0
    for mode in modes:
        if mode not in mode_npzs:
            print(f"  merge skip {mode}: no genie-{mode} NPZ")
            continue
        npz = mode_npzs[mode]
        knobs = GENIE_GROUP_KNOBS.get(mode, [])
        print(f"  merging {mode} ({len(knobs)} knobs)")
        for var_name, row in genie_blob.items():
            if not isinstance(row, dict):
                continue
            by_knob = genie_group_by_knob(npz, var_name)
            if not by_knob:
                continue
            for kn in knobs:
                if kn not in by_knob:
                    continue
                pack = by_knob[kn]
                if not isinstance(pack, dict):
                    continue
                rate_cf = (pack.get("rate") or {}).get("cov_frac")
                xsec_cf = (pack.get("xsec") or {}).get("cov_frac")
                if rate_cf is not None:
                    cf = np.asarray(rate_cf, dtype=np.float64)
                    rk = f"{kn}_rate"
                    if rk not in row:
                        row[rk] = cf.copy()
                        if "genie_rate" not in row:
                            row["genie_rate"] = cf.copy()
                        else:
                            row["genie_rate"] = np.asarray(row["genie_rate"], dtype=np.float64) + cf
                        n_added += 1
                if xsec_cf is not None:
                    cf = np.asarray(xsec_cf, dtype=np.float64)
                    if kn not in row:
                        row[kn] = cf.copy()
                        if "genie" not in row:
                            row["genie"] = cf.copy()
                        else:
                            row["genie"] = np.asarray(row["genie"], dtype=np.float64) + cf
                        n_added += 1
    return n_added


def genie_combined_from_mode_npzs(genie_mode_npzs, var_name, kind):
    """Sum per-mode total ``cov_frac`` matrices (independent interaction modes)."""
    parts = []
    for npz in genie_mode_npzs.values():
        by_knob = genie_group_by_knob(npz, var_name)
        pack = genie_group_knob_covs(by_knob)
        if pack is None:
            continue
        if kind == "rate":
            m = pack["rate_total"]
        else:
            m = pack["xsec_total"]
        if m is not None:
            parts.append(m)
    if not parts:
        return None
    return _sum_cov_frac_matrices(parts)


def genie_pickle_mode_cov_frac(genie_blob, var_name, mode, kind):
    """Sum per-knob ``cov_frac`` from the GENIE pickle for one interaction mode.

    Returns ``(cov_frac, None)`` on success or ``(None, error_message)``."""
    from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS

    gd = genie_var_dict(genie_blob, var_name)
    if gd is None:
        return None, f"variable {var_name!r} not in GENIE pickle"
    knobs = set(GENIE_GROUP_KNOBS.get(mode, []))
    if not knobs:
        return None, f"no knobs registered for GENIE mode {mode!r}"
    parts = []
    matched_keys = []
    for k, v in gd.items():
        if not isinstance(v, np.ndarray) or v.ndim != 2:
            continue
        if k in ("genie", "genie_rate"):
            continue
        if kind == "rate":
            if not str(k).endswith("_rate"):
                continue
            base = str(k)[: -len("_rate")]
        else:
            if str(k).endswith("_rate"):
                continue
            base = str(k)
        if base not in knobs:
            continue
        parts.append(np.asarray(v, dtype=np.float64))
        matched_keys.append(k)
    if not parts:
        gd_mat = sorted(
            k
            for k, v in gd.items()
            if isinstance(v, np.ndarray) and getattr(v, "ndim", 0) == 2
        )
        return (
            None,
            f"no {kind} matrices for mode {mode!r} / {var_name!r} "
            f"(expected {len(knobs)} knob(s), pickle matrix keys: {gd_mat})",
        )
    return _sum_cov_frac_matrices(parts), None


def genie_mode_cov_frac(genie_mode_npz, var_name, kind):
    by_knob = genie_group_by_knob(genie_mode_npz, var_name)
    pack = genie_group_knob_covs(by_knob)
    if pack is None:
        return None
    return genie_category_cov_frac(pack, kind)


def plot_genie_mode_comparison(ax, var_config, genie_blob, genie_mode_npz, mode, kind):
    """Compare GENIE pickle (mode knobs only) vs ``genie-<mode>`` NPZ total.

    Returns ``(True, None)`` if both curves were drawn, else ``(False, error_message)``."""
    vsn = var_config.var_save_name
    bins = var_config.bins
    bc = var_config.bin_centers
    pct_list = []

    from_pickle, err_pkl = genie_pickle_mode_cov_frac(genie_blob, vsn, mode, kind)
    from_npz = genie_mode_cov_frac(genie_mode_npz, vsn, kind)

    if from_pickle is None:
        return False, err_pkl or f"pickle: no {kind} data for {mode}/{vsn}"
    if from_npz is None:
        return False, f"NPZ genie-{mode}: no {kind} data for {vsn}"

    w_pkl = frac_weights_for_plot(from_pickle, var_config)
    pct_list.append(w_pkl)
    ax.hist(
        bc,
        bins=bins,
        weights=w_pkl,
        histtype="step",
        linewidth=2,
        color="C0",
        label=f"pickle ({mode})",
    )

    w_npz = frac_weights_for_plot(from_npz, var_config)
    pct_list.append(w_npz)
    ax.hist(
        bc,
        bins=bins,
        weights=w_npz,
        histtype="step",
        linewidth=2,
        color="C1",
        linestyle="--",
        label=f"genie-{mode} NPZ",
    )

    diff = np.max(np.abs(w_pkl - w_npz))
    ax.text(
        0.02,
        0.98,
        f"max |Δ| = {diff:.4g} %",
        transform=ax.transAxes,
        va="top",
        fontsize=10,
    )

    style_uncertainty_axis(ax, var_config, pct_list, legend_ncol=2)
    return True, None


def plot_genie_total_comparison(ax, var_config, genie_blob, genie_mode_npzs, kind):
    """Compare GENIE total from ``GENIE/cov_mat_dict.pkl`` vs sum of ``genie-*`` directories."""
    vsn = var_config.var_save_name
    bins = var_config.bins
    bc = var_config.bin_centers
    pct_list = []

    from_pickle = None
    if genie_blob is not None:
        gd = genie_var_dict(genie_blob, vsn)
        gp = genie_knob_covs(gd)
        if gp is not None:
            from_pickle = genie_category_cov_frac(gp, kind)

    from_modes = genie_combined_from_mode_npzs(genie_mode_npzs, vsn, kind)

    if from_pickle is not None:
        w = frac_weights_for_plot(from_pickle, var_config)
        pct_list.append(w)
        ax.hist(
            bc,
            bins=bins,
            weights=w,
            histtype="step",
            linewidth=2,
            color="C0",
            label="GENIE/ cov_mat_dict.pkl",
        )
    if from_modes is not None:
        w = frac_weights_for_plot(from_modes, var_config)
        pct_list.append(w)
        ax.hist(
            bc,
            bins=bins,
            weights=w,
            histtype="step",
            linewidth=2,
            color="C1",
            linestyle="--",
            label="Sum genie-* dirs",
        )

    if from_pickle is not None and from_modes is not None:
        diff = np.max(np.abs(frac_unc_pct(from_pickle) - frac_unc_pct(from_modes)))
        ax.text(
            0.02,
            0.98,
            f"max |Δ| = {diff:.4g} %",
            transform=ax.transAxes,
            va="top",
            fontsize=10,
        )

    if not pct_list:
        ax.text(0.5, 0.5, "No GENIE data", transform=ax.transAxes, ha="center", va="center")
        return

    style_uncertainty_axis(
        ax,
        var_config,
        pct_list,
    )


def detector_by_tag_dict(det_npz, var_name):
    """Return {tag_label: cov_frac} plus combined total from ``detector`` key.

    Prefer ``detector_by_wiremod`` (Flux/G4 ``*_by_knob``-style nesting); fall back
    to legacy ``detector-<tag>`` top-level keys. Missing ``var_name`` returns ``({}, None)``."""
    z = dict(det_npz)
    total = None
    combined = z.get("detector")
    if combined is not None:
        comb_item = combined.item() if hasattr(combined, "item") else combined
        if isinstance(comb_item, dict) and var_name in comb_item:
            total = comb_item[var_name]["cov_frac"]
    dbw = z.get("detector_by_wiremod")
    if dbw is not None:
        cell = dbw.item() if hasattr(dbw, "item") else dbw
        if isinstance(cell, dict) and var_name in cell:
            per_var = cell[var_name]
            out = {tag: pack["cov_frac"] for tag, pack in sorted(per_var.items())}
            if total is None and out:
                total = _sum_cov_frac_matrices(out.values())
            return out, total
    out = {}
    for k in sorted(z.keys()):
        if not k.startswith("detector-"):
            continue
        tag = k[len("detector-") :]
        item = z[k].item() if hasattr(z[k], "item") else z[k]
        if var_name not in item:
            continue
        out[tag] = item[var_name]["cov_frac"]
    if total is None and out:
        total = _sum_cov_frac_matrices(out.values())
    return out, total


def genie_var_dict(genie_blob, var_name):
    if genie_blob is None or var_name not in genie_blob:
        return None
    return genie_blob[var_name]


def genie_knob_covs(gd):
    """Split pickle entry into per-knob rate/xsec cov_frac dicts and totals."""
    if gd is None:
        return None
    rate_parts, xsec_parts = {}, {}
    rate_total = xsec_total = None
    for k, v in gd.items():
        if not isinstance(v, np.ndarray) or v.ndim != 2:
            continue
        if k == "genie":
            xsec_total = v
        elif k == "genie_rate":
            rate_total = v
        elif k.endswith("_rate"):
            rate_parts[k[: -len("_rate")]] = v
        else:
            xsec_parts[k] = v
    return {
        "rate_parts": rate_parts,
        "xsec_parts": xsec_parts,
        "rate_total": rate_total,
        "xsec_total": xsec_total,
    }


def _sum_cov_frac_matrices(matrices):
    """Sum square fractional covariance matrices (independent knobs)."""
    total = None
    for m in matrices:
        arr = np.asarray(m, dtype=np.float64)
        total = arr if total is None else total + arr
    return total


def genie_category_cov_frac(genie_pack, kind):
    """Combined GENIE fractional covariance for ``rate`` or ``xsec`` (total matrix or sum of per-knob parts)."""
    if genie_pack is None:
        return None
    if kind == "rate":
        if genie_pack["rate_total"] is not None:
            return np.asarray(genie_pack["rate_total"], dtype=np.float64)
        parts = genie_pack.get("rate_parts") or {}
        if not parts:
            return None
        return _sum_cov_frac_matrices(parts.values())
    if genie_pack["xsec_total"] is not None:
        return np.asarray(genie_pack["xsec_total"], dtype=np.float64)
    parts = genie_pack.get("xsec_parts") or {}
    if not parts:
        return None
    return _sum_cov_frac_matrices(parts.values())


def plot_genie_rate_breakdown(
    ax,
    var_config,
    genie_pack,
    knob_sort_scores=None,
    label_fn=None,
    max_knobs=None,
):
    rp_all = genie_pack.get("rate_parts") or {}
    total = genie_pack.get("rate_total")
    if total is None and rp_all:
        total = _sum_cov_frac_matrices(rp_all.values())
    rp = rp_all
    if max_knobs is not None and rp_all:
        top = _top_knob_keys(rp_all, knob_sort_scores, k=max_knobs)
        rp = {k: rp_all[k] for k in top}
    if not rp and total is None:
        return
    by_knob = {kn: {"cov_frac": m} for kn, m in rp.items()}
    plot_knob_breakdown(
        ax,
        var_config,
        by_knob,
        total,
        "GENIE — rate (per knob)",
        label_fn=label_fn or genie_knob_display_label,
        knob_sort_scores=knob_sort_scores,
        legend_ncol=4,
    )


def plot_genie_xsec_breakdown(
    ax,
    var_config,
    genie_pack,
    knob_sort_scores=None,
    label_fn=None,
    max_knobs=None,
):
    xp_all = genie_pack.get("xsec_parts") or {}
    total = genie_pack.get("xsec_total")
    if total is None and xp_all:
        total = _sum_cov_frac_matrices(xp_all.values())
    xp = xp_all
    if max_knobs is not None and xp_all:
        top = _top_knob_keys(xp_all, knob_sort_scores, k=max_knobs)
        xp = {k: xp_all[k] for k in top}
    if not xp and total is None:
        return
    by_knob = {kn: {"cov_frac": m} for kn, m in xp.items()}
    plot_knob_breakdown(
        ax,
        var_config,
        by_knob,
        total,
        "GENIE — cross section (per knob)",
        label_fn=label_fn or genie_knob_display_label,
        knob_sort_scores=knob_sort_scores,
        legend_ncol=4,
    )


def _cosmics_cov_frac_for_totals(cosmics_npz, var_name):
    return cosmics_cov_frac(cosmics_npz, var_name)


def _flat_frac_unc_pct(var_config, pct):
    """Uncorrelated flat fractional uncertainty [%] in every bin."""
    n = len(var_config.bin_centers)
    w = np.full(n, float(pct), dtype=np.float64)
    if getattr(var_config, "var_save_name", None) == "integrated" and n:
        w = np.full(n, float(pct))
    return w


def _flat_frac_integrated_variance(nbins, pct):
    """Integrated-rate fractional variance for uncorrelated flat pct%% in all bins."""
    f = pct / 100.0
    return float(nbins * f * f)


TARGETS_FLAT_PCT = 1.0
EXPOSURE_FLAT_PCT = 2.0


def _cosmics_label_for_totals(cosmics_npz, var_name):
    return cosmics_selected_rate_plot_label(cosmics_npz, var_name)


def detector_total_cov_frac(vsn, wiremod_npz=None, sce_npz=None, detector_npz=None):
    """Combined WireMod + SCE fractional covariance (sum of subsystem totals)."""
    totals = []
    for det_npz in (wiremod_npz, sce_npz):
        if det_npz is None:
            continue
        _, tot = detector_by_tag_dict(det_npz, vsn)
        if tot is not None:
            totals.append(np.asarray(tot, dtype=np.float64))
    if totals:
        return _sum_cov_frac_matrices(totals) if len(totals) > 1 else totals[0]
    if detector_npz is not None:
        det_item = dict(detector_npz)["detector"].item()
        if vsn in det_item:
            return np.asarray(det_item[vsn]["cov_frac"], dtype=np.float64)
    return None


def _pct_weights_from_cov(cov_frac, var_config):
    """Per-bin %% weights matching ``plot_knob_breakdown`` (``frac_weights_for_plot``)."""
    if cov_frac is None:
        return None
    if var_config is not None:
        return frac_weights_for_plot(cov_frac, var_config)
    return frac_unc_pct(cov_frac)


def category_total_fracs(
    vsn,
    var_config,
    flux_npz,
    g4_npz,
    detector_npz,
    genie_pack,
    cosmics_npz=None,
    mcstat_npz=None,
    wiremod_npz=None,
    sce_npz=None,
):
    """Per-bin sqrt(diag(C)) in percent for Flux / G4 / MCstat / Detector / Cosmics / GENIE."""
    fu = _pct_weights_from_cov(flux_npz[vsn].item()["flux"]["cov_frac"], var_config)
    gu = _pct_weights_from_cov(g4_npz[vsn].item()["G4"]["cov_frac"], var_config)
    mu = du = gr = gx = cu = None
    mc = mcstat_cov_frac(mcstat_npz, vsn)
    if mc is not None:
        mu = _pct_weights_from_cov(mc, var_config)
    det_cov = detector_total_cov_frac(
        vsn, wiremod_npz=wiremod_npz, sce_npz=sce_npz, detector_npz=detector_npz
    )
    if det_cov is not None:
        du = _pct_weights_from_cov(det_cov, var_config)
    if genie_pack is not None:
        c_rate = genie_category_cov_frac(genie_pack, "rate")
        if c_rate is not None:
            gr = _pct_weights_from_cov(c_rate, var_config)
        c_xsec = genie_category_cov_frac(genie_pack, "xsec")
        if c_xsec is not None:
            gx = _pct_weights_from_cov(c_xsec, var_config)
    if var_config is not None and cosmics_npz is not None:
        cu = cosmics_plot_weights(cosmics_npz, var_config)
    return fu, gu, mu, du, gr, gx, cu


def summed_cov_total_frac(
    vsn,
    flux_npz,
    g4_npz,
    detector_npz,
    genie_pack,
    genie_kind,
    cosmics_npz=None,
    mcstat_npz=None,
    wiremod_npz=None,
    sce_npz=None,
):
    """sqrt(diag(sum of category cov_frac)). genie_kind: 'rate' or 'xsec'."""
    cf = flux_npz[vsn].item()["flux"]["cov_frac"]
    cg = g4_npz[vsn].item()["G4"]["cov_frac"]
    s = np.asarray(cf, dtype=np.float64) + np.asarray(cg, dtype=np.float64)
    mc = mcstat_cov_frac(mcstat_npz, vsn)
    if mc is not None:
        s = s + mc
    det_cov = detector_total_cov_frac(
        vsn, wiremod_npz=wiremod_npz, sce_npz=sce_npz, detector_npz=detector_npz
    )
    if det_cov is not None:
        s = s + det_cov
    cc = _cosmics_cov_frac_for_totals(cosmics_npz, vsn)
    if cc is not None:
        s = s + np.asarray(cc, dtype=np.float64)
    gc = genie_category_cov_frac(genie_pack, genie_kind) if genie_pack is not None else None
    if gc is not None:
        s = s + np.asarray(gc, dtype=np.float64)
    return frac_unc_pct(s)


CATEGORY_TOTALS_PLOT_ORDER = (
    "Flux",
    "GENIE",
    "G4",
    "Detector",
    "Exposure",
    "Targets",
    "Cosmics",
    "MC stat.",
)


def plot_category_totals_single_panel(
    ax,
    var_config,
    flux_npz,
    g4_npz,
    detector_npz,
    genie_pack,
    genie_kind,
    genie_blob=None,
    cosmics_npz=None,
    mcstat_npz=None,
    wiremod_npz=None,
    sce_npz=None,
    data_stat_pct=None,
    data_stat_label="Data stat.",
):
    """One axis: category totals in ``CATEGORY_TOTALS_PLOT_ORDER``, then Total syst. / Data stat."""
    vsn = var_config.var_save_name
    bins = var_config.bins
    bc = var_config.bin_centers
    # Match ``syst_break_genie_all_xsec`` combined breakdown (filters + grouped knobs).
    if genie_blob is not None:
        combined = genie_pickle_combined_breakdown_pack(genie_blob, vsn)
        if combined is not None:
            genie_pack = combined
    fu, gu, mu, du, gr, gx, cu = category_total_fracs(
        vsn,
        var_config,
        flux_npz,
        g4_npz,
        detector_npz,
        genie_pack,
        cosmics_npz=cosmics_npz,
        mcstat_npz=mcstat_npz,
        wiremod_npz=wiremod_npz,
        sce_npz=sce_npz,
    )
    pct_list = []
    by_label = {
        "Flux": fu,
        "G4": gu,
        "Targets": _flat_frac_unc_pct(var_config, TARGETS_FLAT_PCT),
        "Exposure": _flat_frac_unc_pct(var_config, EXPOSURE_FLAT_PCT),
    }
    if genie_kind == "rate" and gr is not None:
        by_label["GENIE"] = gr
    elif genie_kind == "xsec" and gx is not None:
        by_label["GENIE"] = gx
    if du is not None:
        by_label["Detector"] = du
    if cu is not None:
        by_label["Cosmics"] = cu
    if mu is not None:
        by_label["MC stat."] = mu
    plot_labels = [lab for lab in CATEGORY_TOTALS_PLOT_ORDER if lab in by_label]
    colors = _uncertainty_step_colors(len(plot_labels))
    for lab, color in zip(plot_labels, colors):
        w = by_label[lab]
        pct_list.append(w)
        ax.hist(
            bc,
            bins=bins,
            weights=w,
            histtype="step",
            linewidth=2,
            color=color,
            # label=("prelim. detector" if lab == "detector" else lab),
            label=("Detector" if lab == "Detector" else lab),
        )
    grand = summed_cov_total_frac(
        vsn,
        flux_npz,
        g4_npz,
        detector_npz,
        genie_pack,
        genie_kind,
        cosmics_npz=cosmics_npz,
        mcstat_npz=mcstat_npz,
        wiremod_npz=wiremod_npz,
        sce_npz=sce_npz,
    )
    grand_frac = np.asarray(grand, dtype=np.float64) / 100.0
    grand_frac = np.sqrt(
        grand_frac ** 2
        + (TARGETS_FLAT_PCT / 100.0) ** 2
        + (EXPOSURE_FLAT_PCT / 100.0) ** 2
    )
    grand = 100.0 * grand_frac
    pct_list.append(grand)
    ax.hist(
        bc,
        bins=bins,
        weights=grand,
        histtype="step",
        linewidth=2,
        color="k",
        label="Total syst.",
    )
    # ax.set_ylim(0, 16)

    if data_stat_pct is not None:
        pct_list.append(data_stat_pct)
        ax.hist(
            bc,
            bins=bins,
            weights=data_stat_pct,
            histtype="step",
            linewidth=2,
            color="k",
            linestyle=":",
            label=data_stat_label,
        )
    style_uncertainty_axis(ax, var_config, pct_list, legend_fontsize=12)


def _integrated_by_knob_scores(npz, by_knob_key):
    if "integrated" not in npz:
        return {}
    return _integrated_knob_scores(npz["integrated"].item().get(by_knob_key))


def plot_all_syst_breakdowns_for_var(
    var_config,
    flux_npz,
    g4_npz,
    detector_npz=None,
    genie_blob=None,
    cosmics_npz=None,
    mcstat_npz=None,
    wiremod_npz=None,
    sce_npz=None,
    out_dir=None,
    show=True,
):
    """Write/show all per-category breakdown figures for one variable (fixed 8×5 inch canvas)."""
    vsn = var_config.var_save_name

    flux_int_scores = _integrated_by_knob_scores(flux_npz, "flux_by_knob")
    g4_int_scores = _integrated_by_knob_scores(g4_npz, "G4_by_knob")
    gd_int = genie_var_dict(genie_blob, "integrated")
    genie_pack_int = genie_knob_covs(gd_int)
    genie_rate_int_scores = (
        _integrated_matrix_scores(genie_pack_int["rate_parts"]) if genie_pack_int else {}
    )
    genie_xsec_int_scores = (
        _integrated_matrix_scores(genie_pack_int["xsec_parts"]) if genie_pack_int else {}
    )

    def _finish(fig, filename):
        if out_dir:
            save_breakdown_figure(fig, Path(out_dir) / filename)
        elif show:
            _lock_breakdown_figure_size(fig)

    fig, ax = make_breakdown_figure()
    cell = flux_npz[vsn].item()
    plot_knob_breakdown(
        ax, var_config, cell.get("flux_by_knob"), cell["flux"]["cov_frac"],
        "", knob_sort_scores=flux_int_scores,
    )
    _finish(fig, f"syst_break_flux__{vsn}.png")

    fig, ax = make_breakdown_figure()
    cell = g4_npz[vsn].item()
    plot_knob_breakdown(
        ax, var_config, cell.get("G4_by_knob"), cell["G4"]["cov_frac"],
        "", knob_sort_scores=g4_int_scores,
    )
    _finish(fig, f"syst_break_g4__{vsn}.png")

    if cosmics_npz is not None and cosmics_cov_frac(cosmics_npz, vsn) is not None:
        fig, ax = make_breakdown_figure()
        plot_cosmics_breakdown(ax, var_config, cosmics_npz)
        _finish(fig, f"syst_break_cosmics__{vsn}.png")

    if detector_npz is not None:
        fig, ax = make_breakdown_figure()
        plot_detector_combined_breakdown(
            ax, var_config, wiremod_npz=detector_npz, sce_npz=None,
        )
        _finish(fig, f"syst_break_detector__{vsn}.png")

    genie_pack = None
    if genie_blob is not None:
        gd = genie_var_dict(genie_blob, vsn)
        genie_pack = genie_knob_covs(gd)
        if genie_pack is not None:
            if genie_pack["rate_parts"] or genie_pack["rate_total"] is not None:
                fig_r, ax_r = make_breakdown_figure()
                plot_genie_rate_breakdown(
                    ax_r, var_config, genie_pack, knob_sort_scores=genie_rate_int_scores,
                )
                _finish(fig_r, f"syst_break_genie_rate__{vsn}.png")
            if genie_pack["xsec_parts"] or genie_pack["xsec_total"] is not None:
                fig_x, ax_x = make_breakdown_figure()
                plot_genie_xsec_breakdown(
                    ax_x, var_config, genie_pack, knob_sort_scores=genie_xsec_int_scores,
                )
                _finish(fig_x, f"syst_break_genie_xsec__{vsn}.png")

    for kind, fname in (("rate", "syst_break_totals_rate__"), ("xsec", "syst_break_totals_xsec__")):
        if kind == "xsec" and genie_blob is None:
            continue
        fig, ax = make_breakdown_figure()
        plot_category_totals_single_panel(
            ax, var_config, flux_npz, g4_npz, detector_npz, genie_pack, kind,
            genie_blob=genie_blob,
            cosmics_npz=cosmics_npz,
            mcstat_npz=mcstat_npz,
            wiremod_npz=wiremod_npz,
            sce_npz=sce_npz,
        )
        _finish(fig, f"{fname}{vsn}.png")




## Breakdown per category


In [ ]:
OUT_DIR

In [ ]:
# ----------------------------
# MCstat uncertainty (finite-MC multisim)
# ----------------------------

for MCSTAT_BREAKDOWN_VAR in var_configs:
    vsn = MCSTAT_BREAKDOWN_VAR.var_save_name
    if mcstat_npz is None or mcstat_cov_frac(mcstat_npz, vsn) is None:
        print(f"Skipping {vsn}: no MCstat entry")
        continue

    fig, ax = make_breakdown_figure()
    plot_mcstat_breakdown(ax, MCSTAT_BREAKDOWN_VAR, mcstat_npz)
    out_path = OUT_DIR / f"syst_break_mcstat__{vsn}.png"
    save_breakdown_figure(fig, out_path)


In [ ]:
# ----------------------------
# Cosmic uncertainty on selected event rate (contamination-scaled SelectedRate)
# ----------------------------

for COSMIC_BREAKDOWN_VAR in var_configs:
    vsn = COSMIC_BREAKDOWN_VAR.var_save_name
    if cosmics_cov_frac(cosmics_npz, vsn) is None:
        print(f"Skipping {vsn}: no SelectedRate cosmics entry")
        continue

    fig, ax = make_breakdown_figure()
    plot_cosmics_breakdown(ax, COSMIC_BREAKDOWN_VAR, cosmics_npz)
    out_path = OUT_DIR / f"syst_break_cosmics__{vsn}.png"
    save_breakdown_figure(fig, out_path)


In [ ]:
# ----------------------------
# Flux uncertainty breakdown (per knob + combined total)
# ----------------------------

# FLUX_BREAKDOWN_VAR = VariableConfig.all_events()  # or pick any entry from var_configs

for FLUX_BREAKDOWN_VAR in var_configs:

    vsn = FLUX_BREAKDOWN_VAR.var_save_name
    cell = flux_npz[vsn].item()
    flux_int_scores = _integrated_by_knob_scores(flux_npz, "flux_by_knob")

    fig, ax = make_breakdown_figure()
    plot_knob_breakdown(
        ax,
        FLUX_BREAKDOWN_VAR,
        cell.get("flux_by_knob"),
        cell["flux"]["cov_frac"],
        "",
        knob_sort_scores=flux_int_scores,
    )
    out_path = OUT_DIR / f"syst_break_flux__{vsn}.png"
    save_breakdown_figure(fig, out_path)


In [ ]:
out_path

In [ ]:
# ----------------------------
# G4 uncertainty breakdown (per knob + combined total)
# ----------------------------

for G4_BREAKDOWN_VAR in var_configs:

    vsn = G4_BREAKDOWN_VAR.var_save_name
    cell = g4_npz[vsn].item()
    g4_int_scores = _integrated_by_knob_scores(g4_npz, "G4_by_knob")

    fig, ax = make_breakdown_figure()
    plot_knob_breakdown(
        ax,
        G4_BREAKDOWN_VAR,
        cell.get("G4_by_knob"),
        cell["G4"]["cov_frac"],
        "",
        knob_sort_scores=g4_int_scores,
    )
    out_path = OUT_DIR / f"syst_break_g4__{vsn}.png"
    save_breakdown_figure(fig, out_path)


In [ ]:
# ----------------------------
# Detector breakdown (WireMod + SCE on one plot)
# ----------------------------

if wiremod_npz is None and sce_npz is None:
    print("Detector breakdown skipped (no WireMod or SCE NPZ)")
else:
    for DET_BREAKDOWN_VAR in var_configs:
        vsn = DET_BREAKDOWN_VAR.var_save_name
        wm_tags, _ = detector_by_tag_dict(wiremod_npz, vsn) if wiremod_npz is not None else ({}, None)
        sce_tags, _ = detector_by_tag_dict(sce_npz, vsn) if sce_npz is not None else ({}, None)
        if not wm_tags and not sce_tags:
            print(f"Skipping detector {vsn}: no WireMod/SCE tags")
            continue
        fig, ax = make_breakdown_figure()
        plot_detector_combined_breakdown(
            ax,
            DET_BREAKDOWN_VAR,
            wiremod_npz=wiremod_npz,
            sce_npz=sce_npz,
        )
        out_path = OUT_DIR / f"syst_break_detector__{vsn}.png"
        save_breakdown_figure(fig, out_path)


In [ ]:
# WireMod and SCE detector breakdowns are combined in the cell above.
# Re-run ``wiremod.ipynb`` / ``sce.ipynb`` cov cells if ``integrated`` is missing from NPZs.


In [ ]:
# ----------------------------
# GENIE breakdown per interaction mode (from GENIE/cov_mat_dict.pkl)
# ----------------------------

from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_ORDER

GENIE_MODES_TO_MERGE_FROM_NPZ = ("CCQE", "MEC", "RES", "nonRES")

if genie_blob is None:
    print("GENIE breakdown skipped (no GENIE/cov_mat_dict.pkl)")
else:
    # Copy CCQE / MEC / RES / nonRES from genie-* NPZs into the pickle (DIS / Other already there).
    _genie_pkl_path = paths["genie"]
    _mode_npzs = load_genie_mode_npzs(SYST_DISK_ROOT)
    _merge_paths = discover_genie_mode_npz_paths(SYST_DISK_ROOT)
    print("Merging into GENIE pickle from NPZs:", {m: str(_merge_paths[m]) for m in GENIE_MODES_TO_MERGE_FROM_NPZ if m in _merge_paths})
    _n_merged = merge_genie_modes_into_pickle(
        genie_blob, _mode_npzs, GENIE_MODES_TO_MERGE_FROM_NPZ
    )
    if _n_merged:
        with open(_genie_pkl_path, "wb") as _gf:
            pickle.dump(genie_blob, _gf, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Wrote updated GENIE pickle ({_n_merged} new knob blocks): {_genie_pkl_path}")
        print_genie_pickle_key_summary(genie_blob)

    for mode in GENIE_GROUP_ORDER:
        for GENIE_BREAKDOWN_VAR in var_configs:
            vsn = GENIE_BREAKDOWN_VAR.var_save_name
            pack = genie_pickle_mode_pack(genie_blob, vsn, mode)
            if pack is None or not (pack["rate_parts"] or pack["xsec_parts"]):
                print(f"Skipping GENIE {mode} / {vsn} (not in pickle)")
                continue

            pack_int = genie_pickle_mode_pack(genie_blob, "integrated", mode)
            rate_scores = (
                _integrated_matrix_scores(pack_int["rate_parts"]) if pack_int else {}
            )
            xsec_scores = (
                _integrated_matrix_scores(pack_int["xsec_parts"]) if pack_int else {}
            )

            if pack["rate_parts"] or pack["rate_total"] is not None:
                fig_r, ax_r = make_breakdown_figure()
                plot_genie_rate_breakdown(
                    ax_r, GENIE_BREAKDOWN_VAR, pack, knob_sort_scores=rate_scores
                )
                out_r = OUT_DIR / f"syst_break_genie_{mode}_rate__{vsn}.png"
                save_breakdown_figure(fig_r, out_r)

            if pack["xsec_parts"] or pack["xsec_total"] is not None:
                fig_x, ax_x = make_breakdown_figure()
                plot_genie_xsec_breakdown(
                    ax_x, GENIE_BREAKDOWN_VAR, pack, knob_sort_scores=xsec_scores
                )
                out_x = OUT_DIR / f"syst_break_genie_{mode}_xsec__{vsn}.png"
                save_breakdown_figure(fig_x, out_x)


## GENIE combined breakdown (all knobs, filtered)

Per-knob rate and xsec breakdown over **all** interaction modes in one figure.

**Excluded:** Other FSI (``*_pi`` / ``*_N``); Ar23p ``D_ZExp``; Ar23p ``q0bin5``; Ar23p ``EDepFSI_DecayAngMEC``; Ar23p ``EDepFSI_NormCCMEC``.

**Grouped** (covariances summed per family, one curve each): ``*q0bin*`` → e.g. CRPA, SF; ``*dial*`` → e.g. QEIntf; ZExp ``_b*`` → ZExp; MEC q0-bins split by Martini vs Valencia.

**Display:** per-variable curves show only the **10** knob families with largest integrated fractional variance (ranked separately for rate and xsec); the black **Total** includes all included knobs.


In [ ]:
# ----------------------------
# GENIE combined breakdown — all knobs (filtered exclusions)
# ----------------------------

from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS

if genie_blob is None:
    print("GENIE combined breakdown skipped (no GENIE/cov_mat_dict.pkl)")
else:
    _n_excl_other = sum(
        1
        for kn in GENIE_GROUP_KNOBS.get("Other", [])
        if genie_knob_excluded_from_combined_breakdown(kn, "Other")
    )
    _n_excl_ar23p = sum(
        1
        for kn in GENIE_GROUP_KNOBS.get("Ar23p", [])
        if genie_knob_excluded_from_combined_breakdown(kn, "Ar23p")
    )
    print(
        f"GENIE combined breakdown: excluding {_n_excl_other} Other FSI (*_pi, *_N) knobs, "
        f"{_n_excl_ar23p} Ar23p knobs (D_ZExp, q0bin5, EDepFSI DecayAngMEC, EDepFSI NormCCMEC); "
        "bin/dial/ZExp-b families grouped"
    )

    for GENIE_COMBINED_VAR in var_configs:
        vsn = GENIE_COMBINED_VAR.var_save_name
        pack = genie_pickle_combined_breakdown_pack(genie_blob, vsn)
        if pack is None or not (pack["rate_parts"] or pack["xsec_parts"]):
            print(f"Skipping GENIE combined / {vsn} (no knobs after filters)")
            continue

        pack_int = genie_pickle_combined_breakdown_pack(genie_blob, "integrated")
        rate_scores = (
            _integrated_matrix_scores(pack_int["rate_parts"]) if pack_int else {}
        )
        xsec_scores = (
            _integrated_matrix_scores(pack_int["xsec_parts"]) if pack_int else {}
        )

        if pack["rate_parts"] or pack["rate_total"] is not None:
            fig_r, ax_r = make_breakdown_figure()
            plot_genie_rate_breakdown(
                ax_r,
                GENIE_COMBINED_VAR,
                pack,
                knob_sort_scores=rate_scores,
                label_fn=genie_combined_group_display_label,
                max_knobs=10,
            )
            out_r = OUT_DIR / f"syst_break_genie_all_rate__{vsn}.png"
            save_breakdown_figure(fig_r, out_r)

        if pack["xsec_parts"] or pack["xsec_total"] is not None:
            fig_x, ax_x = make_breakdown_figure()
            plot_genie_xsec_breakdown(
                ax_x,
                GENIE_COMBINED_VAR,
                pack,
                knob_sort_scores=xsec_scores,
                label_fn=genie_combined_group_display_label,
                max_knobs=10,
            )
            out_x = OUT_DIR / f"syst_break_genie_all_xsec__{vsn}.png"
            save_breakdown_figure(fig_x, out_x)


In [ ]:
# ----------------------------
# Save cov / corr / cov_frac matrices (NPZ export)
# ----------------------------
# Writes ``CategorySummary/category_syst_summary.npz`` (per-category ``cov_frac``,
# ``corr``, and ``cov`` when ``summary_mc_cv__<var>.npy`` exists under ``OUT_DIR``).
# Downstream: ``load_category_syst_summary``, ``get_category_summary_syst_unc``, overlay plots.

import shutil

from analysis_village.numucc_1p0pi.syst_category_summary import (
    CAT_COSMICS,
    CAT_FLUX,
    CAT_GENIE_RATE,
    CAT_GENIE_XSEC,
    TOTAL_RATE,
    TOTAL_XSEC,
    export_category_syst_summary,
    load_category_syst_summary,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import category_summary_npz_path


def _nominal_mc_vectors_for_export():
    """Optional per-var nominal MC counts for absolute ``cov`` in the export."""
    out = {}
    for vc in var_configs:
        vsn = vc.var_save_name
        cache_path = OUT_DIR / f"summary_mc_cv__{vsn}.npy"
        if cache_path.is_file():
            out[vsn] = np.load(cache_path)
    return out


CATEGORY_SUMMARY_NPZ = category_summary_npz_path(SYST_DISK_ROOT)
CATEGORY_SUMMARY_OUT = OUT_DIR / "category_syst_summary.npz"

_nominal_mc_by_var = _nominal_mc_vectors_for_export()
if _nominal_mc_by_var:
    print("Export includes absolute cov for:", sorted(_nominal_mc_by_var.keys()))
else:
    print(
        "Export: cov_frac + corr only (no summary_mc_cv__*.npy under OUT_DIR; "
        "save from signal_hists to add absolute cov)."
    )

export_manifest = export_category_syst_summary(
    str(CATEGORY_SUMMARY_NPZ),
    var_configs,
    flux_npz=flux_npz,
    g4_npz=g4_npz,
    cosmics_npz=cosmics_npz,
    detector_npz=detector_npz,
    wiremod_npz=wiremod_npz,
    sce_npz=sce_npz,
    mcstat_npz=mcstat_npz,
    genie_blob=genie_blob,
    syst_disk_root=SYST_DISK_ROOT,
    nominal_mc_by_var=_nominal_mc_by_var,
)
shutil.copy2(CATEGORY_SUMMARY_NPZ, CATEGORY_SUMMARY_OUT)
print("Wrote", CATEGORY_SUMMARY_NPZ)
print("Copied to", CATEGORY_SUMMARY_OUT)
print("Variables exported:", len(export_manifest["variables"]))
if export_manifest.get("skipped"):
    print("Skipped:", export_manifest["skipped"])

_summary_check = load_category_syst_summary(str(CATEGORY_SUMMARY_NPZ))
_check_vsn = VariableConfig.muon_momentum().var_save_name
_v = _summary_check["by_var"][_check_vsn]
_flux_blk = _v["categories"][CAT_FLUX]
print(f"Check {_check_vsn}: flux cov_frac shape = {_flux_blk['cov_frac'].shape}")
print(f"  flux corr diagonal = {np.diag(_flux_blk['corr'])}")
if "cov" in _flux_blk:
    print(f"  flux cov diagonal = {np.diag(_flux_blk['cov'])}")
print(
    f"  {CAT_COSMICS} mean % = {_v['categories'][CAT_COSMICS]['frac_unc_pct'].mean():.3f}"
)
print(
    f"  {CAT_GENIE_RATE} mean % = {_v['categories'][CAT_GENIE_RATE]['frac_unc_pct'].mean():.3f}, "
    f"{CAT_GENIE_XSEC} mean % = {_v['categories'][CAT_GENIE_XSEC]['frac_unc_pct'].mean():.3f}"
)
print(
    f"  {TOTAL_RATE} mean % = {_v[TOTAL_RATE]['frac_unc_pct'].mean():.3f} "
    f"(data_mc_comparison, data_driven_validation)"
)
print(
    f"  {TOTAL_XSEC} mean % = {_v[TOTAL_XSEC]['frac_unc_pct'].mean():.3f} (unfolding-data)"
)


## Total systematic summary

For every variable in ``var_configs``: category-totals **breakdown histogram** includes **Flux, G4, MC stat., Detector** (WireMod + SCE combined when available), **cosmics** (contamination-scaled ``SelectedRate``), **Targets**, **Exposure**, **GENIE xsec**, and a black **Total Syst.** curve (``sqrt(diag(sum of category C))``). A second figure overlays **data statistical uncertainty** (dotted black curve) from beam-quality-cut data (``data_mc_comparison.ipynb``). Per-category and **total** fractional **covariance / correlation** heatmaps are written for every category in the summary. Norm vs shape **total** matrices use ``Matrix_Decomp`` (``wienersvd``, matching ``utils``).

**Export:** the **Save cov / corr / cov_frac matrices** cell writes ``CategorySummary/category_syst_summary.npz`` (plus manifest JSON) under ``SYST_DISK_ROOT`` and a copy under ``OUT_DIR``. Each category block stores ``cov_frac``, ``corr``, and ``cov`` (absolute covariance when ``OUT_DIR/summary_mc_cv__<var>.npy`` exists). Summed totals: ``total_rate`` (GENIE **rate**) and ``total_xsec`` (GENIE **xsec**). Load with ``load_category_syst_summary()``; use ``total_cov_frac(..., kind='rate'|'xsec')`` or ``get_category_summary_syst_unc`` in ``utils``.

Nominal MC per bin for the decomposition: set ``SUMMARY_MC_VECTOR``, or save ``OUT_DIR/summary_mc_cv__<var>.npy`` from ``signal_hists(...)["nevts_sel_reco"]``.


In [ ]:
from analysis_village.unfolding.wienersvd import Matrix_Decomp
from analysis_village.numucc_1p0pi.utils import plot_heatmap
from pyanalib.split_df_helpers import load_dfs

# Cosmics in summary: contamination-scaled SelectedRate (same as syst_break_cosmics__*).
# Optional: per-bin nominal MC selected reco counts (length nbins). If None, uses cache or unit vector.
SUMMARY_MC_VECTOR = None

# Beam-quality-cut data (same paths as data_mc_comparison.ipynb)
DFS_ROOT = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"
QUALITY_DF_PATH = (
    Path(DFS_ROOT)
    / "2026_05_16_230705__sel_mup-data-1e20/merged_perTPC"
    / "beam_data_1e20_qualitycut.df"
)
DATA_STAT_SCALE = 1.0

quality_dfs = load_dfs(
    str(QUALITY_DF_PATH),
    keys2load=["hdr", "trigger", "evt_good"],
    n_max_concat=10,
)
data_evt_df = quality_dfs["evt_good"]
data_hdr_df = quality_dfs["hdr"].join(quality_dfs["trigger"])
data_evt_df[("mc", "iscc")] = 999
print(
    f"Loaded quality-cut data: {len(data_evt_df):,} evt_good rows from {QUALITY_DF_PATH}"
)

_summary_mc_cache = {}


def _to_corr(cov):
    cov = np.asarray(cov, dtype=np.float64)
    d = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    den = np.outer(d, d)
    out = np.zeros_like(cov)
    m = den > 0
    out[m] = cov[m] / den[m]
    return np.clip(out, -1.0, 1.0)


def _summary_mc_vector(var_config):
    """Nominal MC prediction per bin for Matrix_Decomp (same as utils plot_syst stack)."""
    vsn = var_config.var_save_name
    if SUMMARY_MC_VECTOR is not None:
        return np.asarray(SUMMARY_MC_VECTOR, dtype=np.float64)
    if vsn in _summary_mc_cache:
        return _summary_mc_cache[vsn]
    cache_path = OUT_DIR / f"summary_mc_cv__{vsn}.npy"
    if cache_path.is_file():
        vec = np.load(cache_path)
        _summary_mc_cache[vsn] = vec
        return vec
    nbins = len(var_config.bins) - 1
    print(
        f"WARNING: no summary_mc_cv__{vsn}.npy — using unit MC vector for norm/shape split "
        "(set SUMMARY_MC_VECTOR or save cache from signal_hists for physical decomposition)."
    )
    vec = np.ones(nbins, dtype=np.float64)
    _summary_mc_cache[vsn] = vec
    return vec


def _frac_cov_norm_shape_split(cov_frac, total_mc):
    """Norm (+ mixed) vs shape fractional blocks via Matrix_Decomp (wienersvd / utils)."""
    total_mc = np.asarray(total_mc, dtype=np.float64)
    cov_frac = np.asarray(cov_frac, dtype=np.float64)
    abs_cov = cov_frac * np.outer(total_mc, total_mc)
    cov_norm, cov_mixed, cov_shape = Matrix_Decomp(total_mc, abs_cov)
    den = np.outer(total_mc, total_mc)
    with np.errstate(divide="ignore", invalid="ignore"):
        f_norm = np.where(den > 0, (cov_norm + cov_mixed) / den, 0.0)
        f_shape = np.where(den > 0, cov_shape / den, 0.0)
    return f_norm, f_shape


def _category_summary_pct(cov_frac, var_config=None):
    """Per-bin uncertainty [%]; uses ``frac_weights_for_plot`` when *var_config* is set."""
    if var_config is not None:
        w = frac_weights_for_plot(cov_frac, var_config)
    else:
        w = frac_unc_pct(cov_frac)
    int_var = float(integrated_rate_frac_variance(cov_frac))
    return {
        "per_bin": w,
        "mean_pct": float(np.mean(w)),
        "max_pct": float(np.max(w)),
        "integrated_pct": 100.0 * np.sqrt(max(int_var, 0.0)),
    }


def _plot_summary_heatmap(mat, kind, var_config, out_path, cmap="viridis", vmin=None, vmax=None):
    xlab = (
        var_config.var_labels[1]
        if getattr(var_config, "var_labels", None)
        else var_config.var_save_name
    )
    # kind = "correlation" if cmap == "bwr" else "covariance"
    plot_heatmap(
        np.asarray(mat, dtype=np.float64),
        var_config.bins,
        plot_labels=[xlab, xlab, kind],
        plot=False,
        save_fig=False,
        leave_open=True,
        approval = "preliminary",
        cmap=cmap,
    )
    fig = plt.gcf()
    fig.set_size_inches(BREAKDOWN_FIGSIZE)
    if vmin is not None or vmax is not None:
        for ax in fig.axes:
            if ax.images:
                ax.images[0].set_clim(
                    vmin if vmin is not None else ax.images[0].get_clim()[0],
                    vmax if vmax is not None else ax.images[0].get_clim()[1],
                )
    save_breakdown_figure(fig, out_path)


def _summary_category_slug(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
    )


def _summary_category_covs(vsn):
    covs = {
        "Flux": np.asarray(flux_npz[vsn].item()["flux"]["cov_frac"], dtype=np.float64),
        "G4": np.asarray(g4_npz[vsn].item()["G4"]["cov_frac"], dtype=np.float64),
    }
    mc = mcstat_cov_frac(mcstat_npz, vsn)
    if mc is not None:
        covs["MC stat."] = mc
    det = detector_total_cov_frac(
        vsn, wiremod_npz=wiremod_npz, sce_npz=sce_npz, detector_npz=detector_npz
    )
    if det is not None:
        covs["Detector"] = det
    cc = _cosmics_cov_frac_for_totals(cosmics_npz, vsn)
    if cc is not None:
        covs[cosmics_selected_rate_plot_label(cosmics_npz, vsn)] = np.asarray(cc, dtype=np.float64)
    gp = genie_pickle_combined_breakdown_pack(genie_blob, vsn) if genie_blob is not None else None
    gc = genie_category_cov_frac(gp, "xsec") if gp is not None else None
    if gc is not None:
        covs["GENIE"] = np.asarray(gc, dtype=np.float64)
    return covs


In [ ]:

var_configs = [
                # VariableConfig.all_events(),
                VariableConfig.muon_momentum(),
                VariableConfig.muon_direction(),
                VariableConfig.proton_momentum(),
                VariableConfig.proton_direction(),
                VariableConfig.tki_del_alpha(),
                VariableConfig.tki_del_phi(),
                VariableConfig.tki_del_Tp(),
                VariableConfig.tki_del_p(),
                VariableConfig.tki_del_Tp_x(),
                VariableConfig.tki_del_Tp_y(),
                # VariableConfig.muon_direction_x(),
                # VariableConfig.muon_direction_y(),
                # VariableConfig.proton_direction_x(),
                # VariableConfig.proton_direction_y(),
                # VariableConfig.opening_angle(),
                # VariableConfig.vertex_x(),
                # VariableConfig.vertex_y(),
                # VariableConfig.vertex_z(),
                ]

In [ ]:
frac_cov_dict = {}
for SUMMARY_VAR in var_configs:
    SUMMARY_VSN = SUMMARY_VAR.var_save_name
    summary_data_stat_pct = data_stat_frac_pct(
        data_evt_df, SUMMARY_VAR, scale=DATA_STAT_SCALE
    )

    summary_covs = _summary_category_covs(SUMMARY_VSN)
    summary_total_cov = _sum_cov_frac_matrices(summary_covs.values())
    summary_mc = _summary_mc_vector(SUMMARY_VAR)

    genie_pack = (
        genie_pickle_combined_breakdown_pack(genie_blob, SUMMARY_VSN)
        if genie_blob is not None
        else None
    )

    # # --- Category-totals breakdown histogram (per-bin %, black = summed covariance) ---
    # fig, ax = make_breakdown_figure()
    # plot_category_totals_single_panel(
    #     ax,
    #     SUMMARY_VAR,
    #     flux_npz,
    #     g4_npz,
    #     detector_npz,
    #     genie_pack,
    #     "xsec",
    #     genie_blob=genie_blob,
    #     cosmics_npz=cosmics_npz,
    #     mcstat_npz=mcstat_npz,
    #     wiremod_npz=wiremod_npz,
    #     sce_npz=sce_npz,
    # )
    # save_breakdown_figure(fig, OUT_DIR / f"syst_break_totals_xsec__{SUMMARY_VSN}.png")

    # Same category totals with data Poisson statistical uncertainty overlaid (dotted).
    fig, ax = make_breakdown_figure()
    plot_category_totals_single_panel(
        ax,
        SUMMARY_VAR,
        flux_npz,
        g4_npz,
        detector_npz,
        genie_pack,
        "xsec",
        genie_blob=genie_blob,
        cosmics_npz=cosmics_npz,
        mcstat_npz=mcstat_npz,
        wiremod_npz=wiremod_npz,
        sce_npz=sce_npz,
        data_stat_pct=summary_data_stat_pct,
    )
    save_breakdown_figure(
        fig, OUT_DIR / f"syst_break_totals_xsec__{SUMMARY_VSN}_with_data_stat.png"
    )

    # print(f"\nCategory totals for {SUMMARY_VSN} (per-bin fractional %, GENIE xsec):")
    # print(f"  {'':14s}  mean%    max%   integ%  (integ = sqrt(1^T C 1); meaningful for 1-bin vars)")
    # for k, c in summary_covs.items():
    #     s = _category_summary_pct(c, SUMMARY_VAR)
    #     print(
    #         f"  {k:14s}: {s['mean_pct']:7.2f}% {s['max_pct']:7.2f}% {s['integrated_pct']:7.2f}%"
    #     )
    # st = _category_summary_pct(summary_total_cov, SUMMARY_VAR)
    # print(
    #     f"  {'Total Systematics':14s}: {st['mean_pct']:7.2f}% {st['max_pct']:7.2f}% {st['integrated_pct']:7.2f}%"
    # )

    # Per-category fractional covariance / correlation matrices
    # for cat_name, cov in summary_covs.items():
    #     slug = _summary_category_slug(cat_name)
    #     _plot_summary_heatmap(
    #         cov,
    #         SUMMARY_VAR,
    #         OUT_DIR / f"syst_{slug}_cov_frac__{SUMMARY_VSN}.png",
    #         cmap="viridis",
    #     )
    #     _plot_summary_heatmap(
    #         _to_corr(cov),
    #         SUMMARY_VAR,
    #         OUT_DIR / f"syst_{slug}_corr__{SUMMARY_VSN}.png",
    #         cmap="bwr",
    #         vmin=-1.0,
    #         vmax=1.0,
    #     )

    # Total (summed categories)
    _plot_summary_heatmap(
        summary_total_cov,
        "Covariance",
        SUMMARY_VAR,
        OUT_DIR / f"syst_total_cov_frac__{SUMMARY_VSN}.png",
        cmap="viridis",
    )
    frac_cov_dict[SUMMARY_VSN] = summary_total_cov
    _plot_summary_heatmap(
        _to_corr(summary_total_cov),
        "Correlation",
        SUMMARY_VAR,
        OUT_DIR / f"syst_total_corr__{SUMMARY_VSN}.png",
        cmap="viridis",
        vmin=-1.0,
        vmax=1.0,
    )

    # norm_covs = {k: _frac_cov_norm_shape_split(c, summary_mc)[0] for k, c in summary_covs.items()}
    # shape_covs = {k: _frac_cov_norm_shape_split(c, summary_mc)[1] for k, c in summary_covs.items()}
    # norm_total_cov = _sum_cov_frac_matrices(norm_covs.values())
    # shape_total_cov = _sum_cov_frac_matrices(shape_covs.values())

    # print(f"\nNormalization (+ mixed) totals for {SUMMARY_VSN} (per-bin %, Matrix_Decomp):")
    # for k, c in norm_covs.items():
    #     s = _category_summary_pct(c, SUMMARY_VAR)
    #     print(
    #         f"  {k:14s}: {s['mean_pct']:7.2f}% {s['max_pct']:7.2f}% {s['integrated_pct']:7.2f}%"
    #     )
    # nt = _category_summary_pct(norm_total_cov, SUMMARY_VAR)
    # print(
    #     f"  {'Total Systematics':14s}: {nt['mean_pct']:7.2f}% {nt['max_pct']:7.2f}% {nt['integrated_pct']:7.2f}%"
    # )

    # print(f"\nShape-only totals for {SUMMARY_VSN} (per-bin %, Matrix_Decomp):")
    # for k, c in shape_covs.items():
    #     s = _category_summary_pct(c, SUMMARY_VAR)
    #     print(
    #         f"  {k:14s}: {s['mean_pct']:7.2f}% {s['max_pct']:7.2f}% {s['integrated_pct']:7.2f}%"
    #     )
    # st_shape = _category_summary_pct(shape_total_cov, SUMMARY_VAR)
    # print(
    #     f"  {'Total Systematics':14s}: {st_shape['mean_pct']:7.2f}% {st_shape['max_pct']:7.2f}% {st_shape['integrated_pct']:7.2f}%"
    # )

    # _plot_summary_heatmap(
    #     norm_total_cov,
    #     SUMMARY_VAR,
    #     OUT_DIR / f"syst_total_norm_cov_frac__{SUMMARY_VSN}.png",
    #     cmap="viridis",
    # )
    # _plot_summary_heatmap(
    #     _to_corr(norm_total_cov),
    #     SUMMARY_VAR,
    #     OUT_DIR / f"syst_total_norm_corr__{SUMMARY_VSN}.png",
    #     cmap="bwr",
    #     vmin=-1.0,
    #     vmax=1.0,
    # )
    # _plot_summary_heatmap(
    #     shape_total_cov,
    #     SUMMARY_VAR,
    #     OUT_DIR / f"syst_total_shape_cov_frac__{SUMMARY_VSN}.png",
    #     cmap="viridis",
    # )
    # _plot_summary_heatmap(
    #     _to_corr(shape_total_cov),
    #     SUMMARY_VAR,
    #     OUT_DIR / f"syst_total_shape_corr__{SUMMARY_VSN}.png",
    #     cmap="bwr",
    #     vmin=-1.0,
    #     vmax=1.0,
    # )

In [ ]:
import pickle

outdir = "/exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS"
with open(os.path.join(outdir, "frac_cov_dict.pkl"), "wb") as f:
    pickle.dump(frac_cov_dict, f)

In [ ]:
import importlib
import analysis_village.numucc_1p0pi.syst_disk_layout as _sdl
importlib.reload(_sdl)

In [ ]:
# Re-run the "Save cov / corr / cov_frac matrices" cell above after changing inputs.
# (Kept as a stub so older notebook flows that expected a final export cell still work.)
print("Category summary NPZ:", CATEGORY_SUMMARY_NPZ)
print("Copy:", CATEGORY_SUMMARY_OUT)
